In [7]:
#saniyagoutam prn:23070521126
from pathlib import Path
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import requests # Added for downloading the file

# Ensure creditcard.csv is available
file_path = Path.cwd() / "creditcard.csv"
if not file_path.exists():
    print("creditcard.csv not found locally. Attempting to download...")
    download_url = "https://raw.githubusercontent.com/nsethi31/kaggle-credit-card-fraud-detection-data/master/creditcard.csv"
    try:
        response = requests.get(download_url)
        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
        with open(file_path, "wb") as f:
            f.write(response.content)
        print("creditcard.csv downloaded successfully.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading creditcard.csv: {e}")
        print("Please ensure you have an active internet connection or upload the file manually.")
        raise FileNotFoundError("creditcard.csv was not found and could not be downloaded.")

# Find the dataset from the current folder or one of its parent folders
possible_paths = [file_path] # Use the path where we ensured the file exists
possible_paths += [parent / "Practical 10" / "creditcard.csv" for parent in Path.cwd().parents]
data_path = next((path for path in possible_paths if path.exists()), None)
if data_path is None:
    # This case should ideally not be hit if download was successful and `file_path` is in `possible_paths`
    raise FileNotFoundError("creditcard.csv was not found. Please upload it or check the download link.")
data = pd.read_csv(data_path)

# Drop rows where 'Class' column has NaN values
data.dropna(subset=['Class'], inplace=True)

X = data.drop(columns="Class")
y = data["Class"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
# Scale features using only the training data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
# Apply SMOTE only to the training data to avoid test-set leakage
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
print(f"Dataset shape: {data.shape}")
print(f"Fraud transactions before SMOTE: {y.sum()} ({y.mean() * 100:.2f}%)者に: {y_train.value_counts().to_dict()}")

Dataset shape: (87239, 31)
Fraud transactions before SMOTE: 205 (0.23%)者に: {0: 69627, 1: 69627}


In [8]:
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
# Build a simple deep neural network
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")])
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"])
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=2048,
    verbose=1)
# Convert probabilities to class predictions
probabilities = model.predict(X_test, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)
print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=["Genuine", "Fraud"], zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))

Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7109 - loss: 0.5085 - val_accuracy: 0.9017 - val_loss: 0.1425
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9517 - loss: 0.2116 - val_accuracy: 0.9092 - val_loss: 0.1456
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9612 - loss: 0.1266 - val_accuracy: 0.9230 - val_loss: 0.1263
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9695 - loss: 0.0933 - val_accuracy: 0.9533 - val_loss: 0.1054
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9761 - loss: 0.0760 - val_accuracy: 0.9650 - val_loss: 0.0899
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9806 - loss: 0.0641 - val_accuracy: 0.9744 - val_loss: 0.0757
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9830 - loss: 0.0552 - val_accuracy: 0.9796 - val_loss: 0.0630
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9848 - loss: 0.0488 - val_accuracy: 0.9829 - val_loss: